In [1]:
from estnltk import Text
from estnltk.taggers import VabamorfTagger, VabamorfAnalyzer
from estnltk_neural.taggers import StanzaSyntaxTagger
from estnltk.converters import text_to_json
import sys, os
import re
import csv
import pandas as pd
import json
from tqdm import tqdm
import itertools
import sqlite3

/home/kaire/anaconda3/envs/nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pd.set_option('display.max_colwidth', None)
pd.set_option("display.show_dimensions", True)

In [3]:
RESULT_DIR = "../results/"

DATA_FILE = RESULT_DIR+ "n80_examples_large_v01/gpt_v02/" + "gpt_10K_b10_run01.csv"

BM_DIR = "../locative_adverbial/benchmarks/syntax_errors/"
RESULT_FILE = BM_DIR + "gpt_10K_b10_v01_no_kaandepaarid.csv"
RESULT_FILE2 = BM_DIR + "gpt_10K_b10_v01_kaandepaarid.csv"

DB_FILE = "........./drive_data/v33_koondkorpus_transaktsioonid_v04_2.db"


In [4]:
df1 = pd.read_csv(DATA_FILE, encoding="utf-8", sep=",")

### kõikvõimalikud käändepaarid

In [5]:
all_cases = list(df1["morph_case"].unique())
all_case_pairs = list(itertools.permutations(all_cases,2))

In [6]:
all_case_pairs

[('in', 'adit'),
 ('in', 'ill'),
 ('in', 'el'),
 ('in', 'abl'),
 ('in', 'all'),
 ('in', 'ad'),
 ('adit', 'in'),
 ('adit', 'ill'),
 ('adit', 'el'),
 ('adit', 'abl'),
 ('adit', 'all'),
 ('adit', 'ad'),
 ('ill', 'in'),
 ('ill', 'adit'),
 ('ill', 'el'),
 ('ill', 'abl'),
 ('ill', 'all'),
 ('ill', 'ad'),
 ('el', 'in'),
 ('el', 'adit'),
 ('el', 'ill'),
 ('el', 'abl'),
 ('el', 'all'),
 ('el', 'ad'),
 ('abl', 'in'),
 ('abl', 'adit'),
 ('abl', 'ill'),
 ('abl', 'el'),
 ('abl', 'all'),
 ('abl', 'ad'),
 ('all', 'in'),
 ('all', 'adit'),
 ('all', 'ill'),
 ('all', 'el'),
 ('all', 'abl'),
 ('all', 'ad'),
 ('ad', 'in'),
 ('ad', 'adit'),
 ('ad', 'ill'),
 ('ad', 'el'),
 ('ad', 'abl'),
 ('ad', 'all')]

### morfo analüüsi jaoks

In [7]:
morf_tagger = VabamorfAnalyzer(output_layer='morph_analysis')
stanza_tagger = StanzaSyntaxTagger(input_type='morph_extended', input_morph_layer='morph_extended')


In [8]:
def get_analysis(ex):
    txt = Text(ex.iloc[0]["sentence"])
    txt.tag_layer("words")
    txt.tag_layer("sentences")
    morf_tagger.tag(txt)
    txt.tag_layer('morph_extended')
    stanza_tagger.tag( txt )
    return txt

In [9]:

# connecting with database
conn = sqlite3.connect(DB_FILE)
cur = conn.cursor()


### iga paari kohta mõni näide "no" vastuste seast

In [11]:
df = df1[df1["classification2"]=="no"]

In [13]:
pair_dict = {pair: [] for pair in all_case_pairs}

df_data = []


for pair in tqdm(all_case_pairs):
    c1 = pair[0]
    c2 = pair[1]
    # kui paari esimene kääne on peasõna kääne ja järgnev sõna on teises käändes ning transaction row tabelis
    exs1 = df[df["morph_case"]==c1].sample(frac=1)
    
    for i in range(len(exs1)):
        ex = exs1.iloc[[i]]
        text = get_analysis(ex)
        head_id = int(ex.iloc[0]["head_id"])
        head_loc = int(ex.iloc[0]["head_loc"])-1
        next_word_loc = head_loc+1 if head_loc < len(text.words)-1 else None
        
        if next_word_loc is not None:
            #print(text.morph_analysis)
            #print(text.morph_analysis[head_loc])
            #print(text.morph_analysis[next_word_loc])
            next_w_forms = " ".join(text.morph_analysis[next_word_loc].form)
            if c2 == "adit":
                c2 = "adt"
            if c2 in next_w_forms:
                head_w = text.morph_analysis[head_loc].text
                next_w = text.morph_analysis[next_word_loc].text
                # kas järgmine sõna on transaction tabelis eraldi reana
                on_eraldi_rida = False
                if "'" not in next_w:
                    query = f"SELECT * FROM transaction_row where head_id={head_id} and form='{next_w}'"
                    prev_res = pd.read_sql(query, conn)
                    if len(prev_res) != 0:
                        feats = prev_res.iloc[0].feats
                        if c2 in feats:
                            on_eraldi_rida = True
                
                    #print(i, "on õiges käändes", pair, ex["form"], text.morph_analysis[head_loc].form, text.morph_analysis[next_word_loc].form)
                    if on_eraldi_rida and len(pair_dict[pair]) < 4:
                        case = ex.iloc[0]["morph_case"]
                        verb = ex.iloc[0]["verb"]
                        v_c = ex.iloc[0]["verb_compound"] if not pd.isna(ex.iloc[0]["verb_compound"]) else ""
                        pair_dict[pair].append((verb+ " "+v_c, case, head_w, next_w, text.text))
                        df_data.append((ex.iloc[0]["sentence_id"], head_id,pair, verb, v_c,case, "", head_w, next_w, text.text))
                    elif len(pair_dict[pair]) == 4:
                        break
                #break
    
    #print(exs1.head())
    
    
    # kui paari teine kääne on peasõna kääne ja eelnev sõna on esimeses käändes
    exs1 = df[df["morph_case"]==c2].sample(frac=1)
    
    for j in range(len(exs1)):
        ex = exs1.iloc[[j]]
        text = get_analysis(ex)
        head_id = int(ex.iloc[0]["head_id"])
        head_loc = int(ex.iloc[0]["head_loc"])-1
        prev_word_loc = head_loc-1 if head_loc >=1 else None
        
        if prev_word_loc is not None:
            #print(text.morph_analysis)
            #print(text.morph_analysis[head_loc])
            #print(text.morph_analysis[next_word_loc])
            prev_w_forms = " ".join(text.morph_analysis[prev_word_loc].form)
            if c1 == "adit":
                c1 = "adt"
            if c1 in prev_w_forms:
                head_w = text.morph_analysis[head_loc].text
                prev_w = text.morph_analysis[prev_word_loc].text                
                on_eraldi_rida = False
                if "'" not in prev_w:
                    query = f"SELECT * FROM transaction_row where head_id={head_id} and form='{prev_w}'"
                    prev_res = pd.read_sql(query, conn)
                    if len(prev_res) != 0:
                        feats = prev_res.iloc[0].feats
                        if c1 in feats:
                            on_eraldi_rida = True
                    
                    #print(i, "on õiges käändes", pair, ex["form"], text.morph_analysis[head_loc].form, text.morph_analysis[next_word_loc].form)
                    if on_eraldi_rida and len(pair_dict[pair]) < 4:
                        verb = ex.iloc[0]["verb"]
                        v_c = ex.iloc[0]["verb_compound"] if not pd.isna(ex.iloc[0]["verb_compound"]) else ""
                        pair_dict[pair].append((verb+ " "+v_c,case, head_w, prev_w ,text.text))
                        df_data.append((ex.iloc[0]["sentence_id"], head_id, pair, verb, v_c, case,prev_w, head_w, "", text.text))
                    elif len(pair_dict[pair]) == 4:
                        break
    
    
    #break
    
examples = pd.DataFrame(df_data, columns = ["sentence_id","head_id", "case_pair" ,"verb", "verb_compound", "morph_case", "prev_word", "form", "next_word", "sentence"])


100%|███████████████████████████████████████████| 42/42 [18:54<00:00, 27.01s/it]


In [14]:
examples

,sentence_id,head_id,case_pair,verb,verb_compound,morph_case,prev_word,form,next_word,sentence
0,12081233,19340058,"(in, ill)",kaevama,,in,,plagiaadikahtluses,kohtusse,"Vähe sellest , et tema müüginumbreid raske lüüa on Browni plagiaadikahtluses kohtusse kaevanud kirjanik Lewis Perdue jäi New Yorgi kohtus pika ninaga ."
1,10071166,16155136,"(in, ill)",leidma,,in,suhtumises,koostöösse,,Tegelikult ei ole Siimanni lubatud muutused valitsuskoalitsiooni suhtumises koostöösse opositsiooniga aset leidnud .
2,1096124,1742530,"(in, ill)",looma,,in,Tallinnas,mättasse,,"Kuid nagu me teame - mõõgavennad , kellele see püha plaan mokkamööda polnud , lõid paavsti mehed Tallinnas mättasse - osa neist väidetavalt otse kiriku altari ees ."
3,11211192,17961410,"(in, ill)",toetama,,in,mõttes,kümnesse,,"“ South Park ” on niivõrd s ... animatsiooniga ( loe : tegelaste liikumisega ) , et see ei häirinud mind kui vaatajat no mitte üks põrm - kogu ülejäänud filmireeglistik toetab seda stiilsuse mõttes kümnesse ."
4,2465306,3952095,"(in, el)",olema,ära,in,,osas,korrast,"Ma arvan , et nii Päevalehes kui teisteski lehtedes on asi selles osas korrast ära ."
...,...,...,...,...,...,...,...,...,...,...
61,442019,691424,"(ad, ill)",jooksma,,ad,Sügisjooksul,koomasse,,Sügisjooksul koomasse jooksnud 23aastane allohvitser suri eile kell 14.10.
62,1441197,2293487,"(ad, ill)",maksma,,ad,kuul,dollaritesse,,"Kui Eestis maksis 100 kilovatt-tundi elektrit möödunud kuul dollaritesse ümberarvestatult neli dollarit , siis Leedus maksis elekter 6,5 dollarit ja Lätis 6,25 dollarit ."
63,12303435,19696326,"(ad, ill)",jooksma,,ad,treeningul,paindesse,,Teivastki on Kaseorg vaid ühel treeningul paindesse jooksnud .
64,1626358,2589281,"(ad, ill)",hakkama,,ad,hetkel,kõigesse,,"Aga tagatipuks hakkas ta mingil hetkel kõigesse , mida ametnikud kas koos minuga või minu kaudu talle allakirjutamiseks või otsustamiseks andsid , suhtuma niisuguse umbusuga , et ta hakkaski otsustamist vältima ..."


In [15]:
examples.to_csv(RESULT_FILE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

### iga paari kohta mõni näide "yes" vastuste seast

In [16]:
df = df1[df1["classification2"]=="yes"]

In [18]:
pair_dict = {pair: [] for pair in all_case_pairs}

df_data = []


for pair in tqdm(all_case_pairs):
    c1 = pair[0]
    c2 = pair[1]
    # kui paari esimene kääne on peasõna kääne ja järgnev sõna on teises käändes ning transaction row tabelis
    exs1 = df[df["morph_case"]==c1].sample(frac=1)
    
    for i in range(len(exs1)):
        ex = exs1.iloc[[i]]
        text = get_analysis(ex)
        head_id = int(ex.iloc[0]["head_id"])
        head_loc = int(ex.iloc[0]["head_loc"])-1
        next_word_loc = head_loc+1 if head_loc < len(text.words)-1 else None
        
        if next_word_loc is not None and len(pair_dict[pair]) < 4:
            #print(text.morph_analysis)
            #print(text.morph_analysis[head_loc])
            #print(text.morph_analysis[next_word_loc])
            next_w_forms = " ".join(text.morph_analysis[next_word_loc].form)
            if c2 == "adit":
                c2 = "adt"
            if c2 in next_w_forms:
                head_w = text.morph_analysis[head_loc].text
                next_w = text.morph_analysis[next_word_loc].text
                # kas järgmine sõna on transaction tabelis eraldi reana
                on_eraldi_rida = False
                if "'" not in next_w:
                    query = f"SELECT * FROM transaction_row where head_id={head_id} and form='{next_w}'"
                    prev_res = pd.read_sql(query, conn)
                    if len(prev_res) != 0:
                        feats = prev_res.iloc[0].feats
                        if c2 in feats:
                            on_eraldi_rida = True
                
                    #print(i, "on õiges käändes", pair, ex["form"], text.morph_analysis[head_loc].form, text.morph_analysis[next_word_loc].form)
                    if on_eraldi_rida and len(pair_dict[pair]) < 4:
                        case = ex.iloc[0]["morph_case"]
                        verb = ex.iloc[0]["verb"]
                        v_c = ex.iloc[0]["verb_compound"] if not pd.isna(ex.iloc[0]["verb_compound"]) else ""
                        pair_dict[pair].append((verb+ " "+v_c, case, head_w, next_w, text.text))
                        df_data.append((ex.iloc[0]["sentence_id"], head_id,pair, verb, v_c,case, "", head_w, next_w, text.text))
                    elif len(pair_dict[pair]) == 4:
                        break
                #break
    
    #print(exs1.head())
    
    
    # kui paari teine kääne on peasõna kääne ja eelnev sõna on esimeses käändes
    exs1 = df[df["morph_case"]==c2].sample(frac=1)
    
    for j in range(len(exs1)):
        ex = exs1.iloc[[j]]
        text = get_analysis(ex)
        head_id = int(ex.iloc[0]["head_id"])
        head_loc = int(ex.iloc[0]["head_loc"])-1
        prev_word_loc = head_loc-1 if head_loc >=1 else None
        
        if prev_word_loc is not None  and len(pair_dict[pair]) < 4:
            #print(text.morph_analysis)
            #print(text.morph_analysis[head_loc])
            #print(text.morph_analysis[next_word_loc])
            prev_w_forms = " ".join(text.morph_analysis[prev_word_loc].form)
            if c1 == "adit":
                c1 = "adt"
            if c1 in prev_w_forms:
                head_w = text.morph_analysis[head_loc].text
                prev_w = text.morph_analysis[prev_word_loc].text                
                on_eraldi_rida = False
                if "'" not in prev_w:
                    query = f"SELECT * FROM transaction_row where head_id={head_id} and form='{prev_w}'"
                    prev_res = pd.read_sql(query, conn)
                    if len(prev_res) != 0:
                        feats = prev_res.iloc[0].feats
                        if c1 in feats:
                            on_eraldi_rida = True
                    
                    #print(i, "on õiges käändes", pair, ex["form"], text.morph_analysis[head_loc].form, text.morph_analysis[next_word_loc].form)
                    if on_eraldi_rida and len(pair_dict[pair]) < 4:
                        verb = ex.iloc[0]["verb"]
                        v_c = ex.iloc[0]["verb_compound"] if not pd.isna(ex.iloc[0]["verb_compound"]) else ""
                        pair_dict[pair].append((verb+ " "+v_c,case, head_w, prev_w ,text.text))
                        df_data.append((ex.iloc[0]["sentence_id"], head_id, pair, verb, v_c, case,prev_w, head_w, "", text.text))
                    elif len(pair_dict[pair]) == 4:
                        break
    
    
    #break
    
examples = pd.DataFrame(df_data, columns = ["sentence_id","head_id", "case_pair" ,"verb", "verb_compound", "morph_case", "prev_word", "form", "next_word", "sentence"])


100%|████████████████████████████████████████| 42/42 [1:33:34<00:00, 133.69s/it]


In [19]:
examples

,sentence_id,head_id,case_pair,verb,verb_compound,morph_case,prev_word,form,next_word,sentence
0,9072772,14588411,"(in, ill)",külmuma,,in,,mererannas,jäässe,"Pühapäeva pärastlõunal toimetasid päästetöötajad Tallinna loomaaeda luige , kes oli Maarjamäe juures mererannas jäässe külmunud ."
1,1688341,2687636,"(in, ill)",kaevama,,in,,kohtades,maasse,"Nagu sajad mehed ja naised , kes kuuluvad tema rahvaväe võrgustikku , on Trochmann ja ta naine Carolyn ( 46 ) varunud kuude kaupa relvi ja toiduaineid ning kaevanud neid strateegilistes kohtades maasse , olles valmis katastroofiks , mis nende arvates kindlasti saabub ."
2,8394502,13453478,"(in, ill)",ootama,,in,,DP-laagrites,Ameerikasse,"Võrdluseks , 1940ndatel ootasid Lääne-Saksamaa DP-laagrites Ameerikasse pääsemise luba ka tuhanded eestlastest sõjapõgenikud , ning ookeani ületamise luba tuli oodata kohati aastaid ."
3,559692,884959,"(in, ill)",peitma,,in,,kodus,pesumasinasse,"Saksa politsei nabis kinni põgenenud vangi , kes oli end kodus pesumasinasse peitnud ."
4,14089504,22434207,"(in, el)",omama,,in,,Itaalias,võis-telnutest,"Tegelikult omas Itaalias võis-telnutest parimat isiklikku tippmarki prantslane Jean-Claude Retel , kes mullu heitis ligi 69 meetrit ."
...,...,...,...,...,...,...,...,...,...,...
115,264997,418403,"(ad, abl)",startima,,ad,õhupallil,Loode-Teravmägedelt,,Salomon Andrée ja tema kaks kaaslast startisid spetsiaalselt selleks retkeks ehitatud õhupallil Loode-Teravmägedelt juulis 1897 sihiga lennata üle põhjanaba lähikonna ja jõuda Alaskasse või Põhja-Kanadasse .
116,815132,1300550,"(ad, all)",maanduma,,ad,detsembril,Marsile,,"Uurimisjaam Mars Polar Lander , mis 3. detsembril Marsile maandus ning sealt vee jälgi otsima pidi , on tunnistatud lõplikult kadunuks , teatas USA kosmoseagentuuri ( NASA ) maanduriprojekti juht Richard Cook esmaspäeval ."
117,101707,172002,"(ad, all)",naasma,,ad,kutsel,Saksamaale,,1939. aastal füüreri kutsel Saksamaale naasnud Scheel usaldas oma vara Saksa kultuuromavalitsuse kätesse .
118,152264,253853,"(ad, all)",jooksma,,ad,kiini,Inglismaale,,"“ Narvalasi on haaranud uus emigreerimislaine , ” väidab Niina Antonova oma leheloos , mille pealkirja võiks maakeelde ümber panna kui “ Narvakad jooksevad kiini Inglismaale maapakku ” ."


In [20]:
examples.to_csv(RESULT_FILE2, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

In [21]:
conn.close()